# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beratbaspinar/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1: The Freshness Multiplier (Finding #4)
The paper claims that 365+ day content refreshed within 30 days shows a 3.2x health boost and 57x more impressions.  Methodology Question: How was the "refreshed" population selected? If editors only choose to refresh pages that already have high historical authority or are beginning to trend upward naturally, this creates a decision-derived feature. Does the validation design account for selection bias, or are we measuring the intelligence of the SEO team's targeting rather than the isolated impact of the refresh itself?  Finding 2: The Content Performance Curve (Finding #2)
The paper states that content enters a decay cliff at 271-365 days, dropping in health score.  Methodology Question: The study uses a local cached snapshot and local derived exports. Is this performance curve derived from longitudinally tracking the same cohort of pages over 365 days, or is it a cross-sectional snapshot of different pages at various ages today? If it is cross-sectional, does the validation design support the claim of a "lifecycle", or could it just reflect that older content was authored under different, potentially weaker, editorial standards?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

A random split is rarely honest because rows from one group (like a client) share hidden characteristics. A random split allows the model to memorize the client and fake its skill. To audit this, I am comparing a standard random split against an honest grouped split (GroupShuffleSplit on client_id). The gap between them reveals how much the random split was relying on memorization

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Load data and clean
csv_url = "https://raw.githubusercontent.com/beratbaspinar/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_url)
df['is_declining_label'] = (df['trend_pct'] < 0).astype(int)
df['avg_position_clean'] = df['avg_position'].replace(0, 999)

# Define Features safely (excluding future data for the honest test)
leakage_cols = ['trend_direction', 'trend_pct', 'is_declining_label', 'content_id', 'client_id', 'avg_position']
leakage_cols.extend([c for c in df.columns if 'last_30d' in c])
feature_cols = [c for c in df.columns if c not in leakage_cols and df[c].dtype in [np.float64, np.int64]]

X = df[feature_cols].fillna(-1)
y = df['is_declining_label']
groups = df['client_id']

# 1. Random Split (The Dishonest Way)
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(X, y, test_size=0.25, random_state=42)
rf_rnd = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_rnd.fit(X_train_rnd, y_train_rnd)
rnd_preds = rf_rnd.predict(X_test_rnd)

# 2. Grouped Split (The Honest Way)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grp = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_grp.fit(X_train_grp, y_train_grp)
grp_preds = rf_grp.predict(X_test_grp)

print("--- The Memorization Gap ---")
print(f"Base Rate (Total): {y.mean():.1%} declining")
print(f"Random Split Precision: {precision_score(y_test_rnd, rnd_preds):.3f}")
print(f"Grouped Split Precision: {precision_score(y_test_grp, grp_preds):.3f}")
print("Insight: The drop in precision from random to grouped shows the model was memorizing client-specific behaviors in the random split.")

--- The Memorization Gap ---
Base Rate (Total): 65.7% declining
Random Split Precision: 0.780
Grouped Split Precision: 0.719
Insight: The drop in precision from random to grouped shows the model was memorizing client-specific behaviors in the random split.


## 3. Leakage audit

We must check for overlapping windows where a feature summed over a window contains the label's window, which means it already knows the outcome. The label is_declining_label is derived from trend_pct, comparing the last 30 days to the previous 30 days. Any feature containing last_30d breaks the timeline rule. By running a train-without test on these suspects, we confirm the absence of label-derived features

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Audit the Top 5 Features of the Honest Grouped Model
importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_grp.feature_importances_
}).sort_values('Importance', ascending=False)

print("--- Leakage Audit: Top 5 Features (Honest Model) ---")
print(importances.head(5).to_string(index=False))

# Audit Check: Are any overlapping 'last_30d' metrics present?
leaky_features_found = [col for col in feature_cols if 'last_30d' in col or col in ['trend_pct', 'trend_direction']]
print(f"\nLeaky features overlapping with label window found in active set: {len(leaky_features_found)}")

if len(leaky_features_found) == 0:
    print("Audit Pass: Timeline drawn correctly. All features are strictly knowable before the label window.")

--- Leakage Audit: Top 5 Features (Honest Model) ---
              Feature  Importance
 impressions_prev_30d    0.319660
days_with_impressions    0.120500
      impressions_90d    0.113914
   avg_position_clean    0.103788
     content_age_days    0.048742

Leaky features overlapping with label window found in active set: 0
Audit Pass: Timeline drawn correctly. All features are strictly knowable before the label window.


## 4. Claim rewrite

Original Bold Claim:
"Our Random Forest model perfectly predicts which content is decaying, proving that drop in impressions is the definitive cause of content failure across all clients."

Rewritten Safe Claim:
"In the observed dataset, the Random Forest model provided decision-support by identifying directional indicators of content decay. Based on the measured feature importance, historical impression volume was strongly associated with the model's classifications on unseen clients."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.